In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [14]:
stocks = ["AAPL", "TSLA"]

data = yf.download(
    stocks, 
    start = "2015-01-01",
    end = "2026-01-01",
    interval = "1d",
    auto_adjust = True # requests prices adjusted for corporote actions like splits and dividends
)

print(data.head())
print(data.shape)

[*********************100%***********************]  2 of 2 completed

Price           Close                  High                   Low             \
Ticker           AAPL       TSLA       AAPL       TSLA       AAPL       TSLA   
Date                                                                           
2015-01-02  24.171755  14.620667  24.638255  14.883333  23.733996  14.217333   
2015-01-05  23.490808  14.006000  24.021425  14.433333  23.305094  13.810667   
2015-01-06  23.493008  14.085333  23.751682  14.280000  23.132630  13.614000   
2015-01-07  23.822437  14.063333  23.921927  14.318667  23.590292  13.985333   
2015-01-08  24.737743  14.041333  24.795227  14.253333  24.032466  14.000667   

Price            Open                Volume            
Ticker           AAPL       TSLA       AAPL      TSLA  
Date                                                   
2015-01-02  24.627199  14.858000  212818400  71466000  
2015-01-05  23.941832  14.303333  257142000  80527500  
2015-01-06  23.554913  14.004000  263188400  93928500  
2015-01-07  23.700837  

In [16]:
prices = data["Close"]

In [17]:
print(prices.head()) #displays firsy 5 rows
print(prices.shape) # returns number of rows and columns
print(prices.isna().sum()) # identifies and counts missing values

Ticker           AAPL       TSLA
Date                            
2015-01-02  24.171755  14.620667
2015-01-05  23.490808  14.006000
2015-01-06  23.493008  14.085333
2015-01-07  23.822437  14.063333
2015-01-08  24.737743  14.041333
(2766, 2)
Ticker
AAPL    0
TSLA    0
dtype: int64


## Logarithmic Daily Returns

In [18]:
returns = np.log(prices / prices.shift(1)) #.shift shifts prices down one day so each days price can be compared with previous days price
returns = returns.dropna()
print(returns.head())
print(returns.shape)

Ticker          AAPL      TSLA
Date                          
2015-01-05 -0.028576 -0.042950
2015-01-06  0.000094  0.005648
2015-01-07  0.013925 -0.001563
2015-01-08  0.037702 -0.001566
2015-01-09  0.001072 -0.018981
(2765, 2)


The 1st observation has no preceeding trading-day price, so its logarithmic return cannot be calculated, why the closing price observation has one extra value. 

In [19]:
print(returns.describe())

Ticker         AAPL         TSLA
count   2765.000000  2765.000000
mean       0.000874     0.001239
std        0.018154     0.036274
min       -0.137708    -0.236518
25%       -0.007309    -0.016677
50%        0.000950     0.001257
75%        0.009964     0.019394
max        0.142617     0.204491


## Sample Covariance Matrix

In [20]:
mean_returns = returns.mean() # calculate mean of each stock
centered_returns = returns - mean_returns
n = len(returns) # no observations
cov_manual = centered_returns.T @ centered_returns / (n-1)

print(cov_manual)

Ticker      AAPL      TSLA
Ticker                    
AAPL    0.000330  0.000286
TSLA    0.000286  0.001316


In [21]:
cov_pandas = returns.cov()
print(cov_pandas)

Ticker      AAPL      TSLA
Ticker                    
AAPL    0.000330  0.000286
TSLA    0.000286  0.001316


In [24]:
print(np.allclose(cov_manual, cov_pandas))

True


## Sample Correlation Matrix

In [25]:
std_returns = returns.std() # sample standard deviation
corr_manual = cov_manual / np.outer(std_returns, std_returns)
print(corr_manual)

Ticker      AAPL      TSLA
Ticker                    
AAPL    1.000000  0.435021
TSLA    0.435021  1.000000


In [26]:
corr_pandas = returns.corr()
print(corr_pandas)

Ticker      AAPL      TSLA
Ticker                    
AAPL    1.000000  0.435021
TSLA    0.435021  1.000000


In [28]:
print(np.allclose(corr_manual, corr_pandas))

True


## Verify Mathematical Properties

In [29]:
print(np.allclose(cov_pandas, cov_pandas.T))
print(np.allclose(corr_pandas, corr_pandas.T))

True
True


They are symmetric

In [32]:
print(np.linalg.eigvalsh(cov_pandas))
print(np.linalg.eigvalsh(corr_pandas))

[0.00025241 0.00139301]
[0.56497926 1.43502074]


Valid, the eigenvalues are non-negative and within numerical tolerance

In [31]:
print((corr_pandas >= -1).all().all())
print((corr_pandas <= 1).all().all())

True
True


Pearson coeffiecent lies between -1 and 1